# 📄 FIR Document Quality Analysis & Zone-wise Visualization
---

This notebook performs:

### ✅ 1. Recursive PDF Quality Analysis
### ✅ 2. Zone-wise Document Score Aggregation
### ✅ 3. Graph Generation
### - Document Score Charts
### - Page Distribution Boxplots
### - Heatmaps of Page Quality
### - Zone Summary Bar Chart

Output report:
- `fir_quality_report.json`
- Plots in `output_graphs/`


In [1]:
import os
import fitz  # PyMuPDF
import cv2
import numpy as np
from pdf2image import convert_from_path
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("Libraries loaded.")

Libraries loaded.


# 📌 Quality Metric Functions
These functions evaluate:
- Sharpness
- Contrast
- Noise
- Skew angle
- Composite quality score

In [2]:
def compute_sharpness(img_gray):
    return cv2.Laplacian(img_gray, cv2.CV_64F).var()

def compute_contrast(img_gray):
    return img_gray.std()

def compute_noise(img_gray):
    mean = np.mean(img_gray)
    std = np.std(img_gray)
    return std / mean if mean != 0 else 0

def compute_skew(img_gray):
    coords = np.column_stack(np.where(img_gray < 200))
    if len(coords) == 0:
        return 0
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    return abs(angle)

def score_page(sharp, contrast, noise, skew):
    sharp_score = min(sharp / 100, 1)
    contrast_score = min(contrast / 80, 1)
    noise_score = min(noise / 0.1, 1)
    skew_score = max(0, 1 - (skew / 10))

    return round(0.35 * sharp_score +
                 0.30 * contrast_score +
                 0.20 * noise_score +
                 0.15 * skew_score, 3)

print("Quality functions ready.")

Quality functions ready.


# 📌 PDF Analysis Function
Converts PDF → images → calculates metrics for each page.

In [8]:
def analyze_pdf(pdf_path):
    print("Analyzing:", pdf_path)
    poppler_path = "/opt/homebrew/bin" 
    pages = convert_from_path(pdf_path, dpi=200,poppler_path=poppler_path)
    print("pages",pages)
    results = []

    for page_num, page in enumerate(pages, start=1):
        img = np.array(page)
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        sharp = compute_sharpness(img_gray)
        contrast = compute_contrast(img_gray)
        noise = compute_noise(img_gray)
        skew = compute_skew(img_gray)
        score = score_page(sharp, contrast, noise, skew)

        results.append({
            "page": page_num,
            "sharpness": sharp,
            "contrast": contrast,
            "noise": noise,
            "skew_angle": skew,
            "quality_score": score
        })

    pdf_score = round(sum(r["quality_score"] for r in results) / len(results), 3)
    return results, pdf_score

# 📌 Zone-wise Recursive Processing
Walks through CZ / NZ / WZ / SZ directories and builds a full JSON report.

In [11]:
def process_all_zones(root_folder="/Users/vidhya/Projects/cb-cid/data/FIR"):
    zones = ["CZ", "NZ", "WZ", "SZ"]
    # zones = [ "SZ"]
    final_output = {}

    for zone in zones:
        zone_path = os.path.join(root_folder, zone)
        if not os.path.isdir(zone_path):
            print(f"   Skipping {zone}, directory not found.")
            continue

        print(f"\nProcessing Zone: {zone}")
        zone_result = {}
        zone_scores = []

        for root, dirs, files in os.walk(zone_path):
            for file in files:
                if file.lower().endswith(".pdf"):
                    pdf_path = os.path.join(root, file)
                    print(f"  - Analyzing: {pdf_path}")

                    try:
                        page_results, pdf_score = analyze_pdf(pdf_path)
                        zone_result[file] = {
                            "path": pdf_path,
                            "document_score": pdf_score,
                            "pages": page_results
                        }
                        zone_scores.append(pdf_score)

                    except Exception as e:
                        zone_result[file] = {"error": str(e)}

        zone_summary_score = round(sum(zone_scores)/len(zone_scores), 3) if zone_scores else 0

        final_output[zone] = {
            "zone_average_score": zone_summary_score,
            "documents": zone_result
        }

    return final_output

print("Zone processor ready.")

Zone processor ready.


# 📌 Run Quality Pipeline & Save JSON Report

In [12]:
report = process_all_zones()

with open("fir_quality_report.json", "w") as f:
    json.dump(report, f, indent=4)

print("\n✔️ FIR Quality Report Saved → fir_quality_report.json")


Processing Zone: CZ
  - Analyzing: /Users/vidhya/Projects/cb-cid/data/FIR/CZ/Trichy CBCID Cr.No.02 of 2022 Suspetious drawing death.pdf
Analyzing: /Users/vidhya/Projects/cb-cid/data/FIR/CZ/Trichy CBCID Cr.No.02 of 2022 Suspetious drawing death.pdf
pages [<PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1654x2339 at 0x17EB54910>, <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1654x2339 at 0x17EB56DD0>, <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1654x2339 at 0x17EB54650>, <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1654x2339 at 0x17EB54A10>]
  - Analyzing: /Users/vidhya/Projects/cb-cid/data/FIR/CZ/Karur Velayutham palayam PS.Cr.No.339 of 2011  Theft case  UN.pdf
Analyzing: /Users/vidhya/Projects/cb-cid/data/FIR/CZ/Karur Velayutham palayam PS.Cr.No.339 of 2011  Theft case  UN.pdf
pages [<PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1653x2339 at 0x17EB54310>, <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1653x2339 at 0x17EB54910>]
  - Analyzing: 

# 📊 Graph Generation Section
Creates:
- Document Score Bar Charts
- Page Score Boxplots
- Heatmaps
- Zone Summary Chart

In [ ]:
os.makedirs("output_graphs", exist_ok=True)

with open("fir_quality_report.json", "r") as f:
    data = json.load(f)

# -------------------------------------------
# 1️⃣ Document Scores – Bar Graph with Labels
# -------------------------------------------
def plot_zone_document_scores(zone_name, zone_data):
    docs = zone_data["documents"]
    names, scores = [], []

    for doc, info in docs.items():
        if "document_score" in info:
            names.append(doc)
            scores.append(info["document_score"])

    if not scores:
        return

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=names, y=scores, palette="viridis")

    # Add value labels on top of bars
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.2f}", 
                ha='center', fontsize=10, fontweight="bold")

    plt.xticks(rotation=45, ha="right")
    plt.title(f"Document Scores – {zone_name}", fontsize=14)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_document_scores.png")
    plt.close()

# --------------------------------------------------
# 2️⃣ Page Score Boxplot with Mean Marker + Label
# --------------------------------------------------
def plot_zone_page_boxplot(zone_name, zone_data):
    pages = []
    for doc, info in zone_data["documents"].items():
        if "pages" in info:
            for p in info["pages"]:
                pages.append(p["quality_score"])

    if not pages:
        return

    plt.figure(figsize=(6, 6))
    sns.boxplot(y=pages, color="skyblue")

    # Add mean line
    mean_val = sum(pages) / len(pages)
    plt.axhline(mean_val, color='red', linestyle='--', linewidth=1.2)
    plt.text(0, mean_val + 0.02, f"Mean: {mean_val:.2f}", 
             color='red', fontsize=10, fontweight="bold")

    plt.title(f"Page Score Distribution – {zone_name}", fontsize=14)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_page_boxplot.png")
    plt.close()

# -----------------------------------------
# 3️⃣ Heatmap with Annotated Page Scores
# -----------------------------------------
def plot_zone_heatmap(zone_name, zone_data):
    rows = []
    docs = zone_data["documents"]

    for doc, info in docs.items():
        if "pages" in info:
            for p in info["pages"]:
                rows.append({
                    "document": doc,
                    "page": p["page"],
                    "score": p["quality_score"]
                })

    if not rows:
        return

    df = pd.DataFrame(rows)
    pivot = df.pivot(index="document", columns="page", values="score")

    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot, annot=True, cmap="YlGnBu", vmin=0, vmax=1,
                fmt=".2f", annot_kws={"size": 9, "weight": "bold"})

    plt.title(f"Page Heatmap – {zone_name}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_heatmap.png")
    plt.close()

# ------------------------------------------------
# 4️⃣ Zone Summary (Bar Plot with Value Labels)
# ------------------------------------------------
def plot_zone_summary(data):
    zones = []
    scores = []

    for zone, info in data.items():
        if isinstance(info.get("zone_average_score"), (float, int)):
            zones.append(zone)
            scores.append(info["zone_average_score"])

    plt.figure(figsize=(8, 6))
    ax = sns.barplot(x=zones, y=scores, palette="coolwarm")

    # Add value labels
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.2f}", 
                ha='center', fontsize=11, fontweight="bold")

    plt.title("Zone Average Quality Scores", fontsize=14)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_summary.png")
    plt.close()


# 📈 Generate All Graphs

In [14]:
for zone_name, zone_data in data.items():
    print(f"Generating graphs for {zone_name}...")
    plot_zone_document_scores(zone_name, zone_data)
    plot_zone_page_boxplot(zone_name, zone_data)
    plot_zone_heatmap(zone_name, zone_data)

plot_zone_summary(data)
print("\n✔️ All graphs saved in output_graphs/")

Generating graphs for CZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=names, y=scores, palette="viridis")


Generating graphs for NZ...
Generating graphs for WZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:23: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Generating graphs for SZ...

✔️ All graphs saved in output_graphs/


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:79: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=zones, y=scores, palette="coolwarm")


In [19]:
os.makedirs("output_graphs", exist_ok=True)

with open("fir_quality_report.json", "r") as f:
    data = json.load(f)

# -------------------------------------------
# 1️⃣ Document Scores – Bar Graph with Labels
# -------------------------------------------
def plot_zone_document_scores(zone_name, zone_data):
    docs = zone_data["documents"]
    names, scores = [], []

    for doc, info in docs.items():
        if "document_score" in info:
            names.append(doc)
            scores.append(info["document_score"])

    if not scores:
        return

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=names, y=scores, palette="viridis")

    # Add value labels on top of bars
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.2f}", 
                ha='center', fontsize=10, fontweight="bold")

    plt.xticks(rotation=45, ha="right")
    plt.title(f"Document Scores – {zone_name}", fontsize=14)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_document_scores.png")
    plt.close()

# --------------------------------------------------
# 2️⃣ Page Score Boxplot with Mean Marker + Label
# --------------------------------------------------
def plot_zone_page_boxplot(zone_name, zone_data):
    pages = []
    for doc, info in zone_data["documents"].items():
        if "pages" in info:
            for p in info["pages"]:
                pages.append(p["quality_score"])

    if not pages:
        return

    plt.figure(figsize=(6, 6))
    sns.boxplot(y=pages, color="skyblue")

    # Add mean line
    mean_val = sum(pages) / len(pages)
    plt.axhline(mean_val, color='red', linestyle='--', linewidth=1.2)
    plt.text(0, mean_val + 0.02, f"Mean: {mean_val:.2f}", 
             color='red', fontsize=10, fontweight="bold")

    plt.title(f"Page Score Distribution – {zone_name}", fontsize=14)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_page_boxplot.png")
    plt.close()

# -----------------------------------------
# 3️⃣ Heatmap with Annotated Page Scores
# -----------------------------------------
def plot_zone_heatmap(zone_name, zone_data):
    rows = []
    docs = zone_data["documents"]

    for doc, info in docs.items():
        if "pages" in info:
            for p in info["pages"]:
                rows.append({
                    "document": doc,
                    "page": p["page"],
                    "score": p["quality_score"]
                })

    if not rows:
        return

    df = pd.DataFrame(rows)
    pivot = df.pivot(index="document", columns="page", values="score")

    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot, annot=True, cmap="YlGnBu", vmin=0, vmax=1,
                fmt=".2f", annot_kws={"size": 9, "weight": "bold"})

    plt.title(f"Page Heatmap – {zone_name}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_heatmap.png")
    plt.close()

# ------------------------------------------------
# 4️⃣ Zone Summary (Bar Plot with Value Labels)
# ------------------------------------------------
def plot_zone_summary(data):
    zones = []
    scores = []

    for zone, info in data.items():
        if isinstance(info.get("zone_average_score"), (float, int)):
            zones.append(zone)
            scores.append(info["zone_average_score"])

    plt.figure(figsize=(8, 6))
    ax = sns.barplot(x=zones, y=scores, palette="coolwarm")

    # Add value labels
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.2f}", 
                ha='center', fontsize=11, fontweight="bold")

    plt.title("Zone Average Quality Scores", fontsize=14)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_summary.png")
    plt.close()


In [17]:
for zone_name, zone_data in data.items():
    print(f"Generating graphs for {zone_name}...")
    plot_zone_document_scores(zone_name, zone_data)
    plot_zone_page_boxplot(zone_name, zone_data)
    plot_zone_heatmap(zone_name, zone_data)

plot_zone_summary(data)
print("\n✔️ All graphs saved in output_graphs/")

Generating graphs for CZ...
Generating graphs for NZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=names, y=scores, palette="viridis")


Generating graphs for WZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:32: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Generating graphs for SZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:108: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=zones, y=scores, palette="coolwarm")



✔️ All graphs saved in output_graphs/


In [21]:
# ================================================
# 🔍 Zone Comparison Charts
# ================================================

# Prepare comparison data
zone_comparison = {
    "zone": [],
    "avg_score": [],
    "min_score": [],
    "max_score": [],
    "doc_count": [],
    "page_count": []
}

print("\n📊 Processing zones for comparison...")
for zone, info in data.items():
    docs = info["documents"]
    all_scores = []
    total_pages = 0
    error_count = 0
    
    print(f"\n  {zone}:")
    for doc, doc_info in docs.items():
        if "document_score" in doc_info:
            all_scores.append(doc_info["document_score"])
            if "pages" in doc_info:
                total_pages += len(doc_info["pages"])
        elif "error" in doc_info:
            error_count += 1
    
    print(f"    ✓ Valid documents: {len(all_scores)}")
    print(f"    ✗ Failed documents (with errors): {error_count}")
    print(f"    Total pages: {total_pages}")
    
    if all_scores:
        zone_comparison["zone"].append(zone)
        zone_comparison["avg_score"].append(info["zone_average_score"])
        zone_comparison["min_score"].append(min(all_scores))
        zone_comparison["max_score"].append(max(all_scores))
        zone_comparison["doc_count"].append(len(all_scores))
        zone_comparison["page_count"].append(total_pages)

comparison_df = pd.DataFrame(zone_comparison)

# 1️⃣ Zone Comparison - Min/Max/Average Scores
plt.figure(figsize=(10, 6))
x = range(len(comparison_df))
width = 0.25

bars1 = plt.bar([i - width for i in x], comparison_df["min_score"], width, label="Min Score", color="lightcoral", alpha=0.8)
bars2 = plt.bar([i for i in x], comparison_df["avg_score"], width, label="Avg Score", color="skyblue", alpha=0.8)
bars3 = plt.bar([i + width for i in x], comparison_df["max_score"], width, label="Max Score", color="lightgreen", alpha=0.8)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Quality Score", fontsize=12, fontweight='bold')
plt.title("Zone Comparison - Min/Average/Max Quality Scores", fontsize=14, fontweight='bold')
plt.xticks(x, comparison_df["zone"])
plt.legend()
plt.ylim(0, 1.1)
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_comparison_scores.png", dpi=300)
plt.close()

# 2️⃣ Zone Comparison - Document Count
plt.figure(figsize=(8, 6))
bars = plt.bar(comparison_df["zone"], comparison_df["doc_count"], color="steelblue", alpha=0.8)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{int(height)}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Number of Documents", fontsize=12, fontweight='bold')
plt.title("Zone Comparison - Document Count", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_comparison_doc_count.png", dpi=300)
plt.close()

# 3️⃣ Zone Comparison - Page Count
plt.figure(figsize=(8, 6))
bars = plt.bar(comparison_df["zone"], comparison_df["page_count"], color="darkorange", alpha=0.8)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{int(height)}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Number of Pages", fontsize=12, fontweight='bold')
plt.title("Zone Comparison - Total Page Count", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_comparison_page_count.png", dpi=300)
plt.close()

# 4️⃣ Zone Comparison - Radar Chart (Multi-metric)
from math import pi

categories = ["Avg Score", "Normalized Doc Count", "Normalized Page Count"]
angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

plt.figure(figsize=(10, 10))
ax = plt.subplot(111, projection='polar')

# Normalize metrics for radar chart
max_docs = comparison_df["doc_count"].max()
max_pages = comparison_df["page_count"].max()

for idx, zone in enumerate(comparison_df["zone"]):
    values = [
        comparison_df.loc[idx, "avg_score"],
        comparison_df.loc[idx, "doc_count"] / max_docs,
        comparison_df.loc[idx, "page_count"] / max_pages
    ]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=zone)
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_title("Zone Comparison - Multi-Metric Radar Chart", fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_comparison_radar.png", dpi=300, bbox_inches='tight')
plt.close()

print("\n✔️ Zone comparison charts generated:")
print("   - zone_comparison_scores.png (Min/Avg/Max comparison)")
print("   - zone_comparison_doc_count.png (Document count)")
print("   - zone_comparison_page_count.png (Page count)")
print("   - zone_comparison_radar.png (Multi-metric radar chart)")
print("\n📊 Comparison Summary:")
print(comparison_df.to_string(index=False))



📊 Processing zones for comparison...

  CZ:
    ✓ Valid documents: 5
    ✗ Failed documents (with errors): 0
    Total pages: 18

  NZ:
    ✓ Valid documents: 7
    ✗ Failed documents (with errors): 0
    Total pages: 23

  WZ:
    ✓ Valid documents: 42
    ✗ Failed documents (with errors): 0
    Total pages: 141

  SZ:
    ✓ Valid documents: 2
    ✗ Failed documents (with errors): 0
    Total pages: 5

✔️ Zone comparison charts generated:
   - zone_comparison_scores.png (Min/Avg/Max comparison)
   - zone_comparison_doc_count.png (Document count)
   - zone_comparison_page_count.png (Page count)
   - zone_comparison_radar.png (Multi-metric radar chart)

📊 Comparison Summary:
zone  avg_score  min_score  max_score  doc_count  page_count
  CZ      0.801      0.751      0.842          5          18
  NZ      0.789      0.681      0.874          7          23
  WZ      0.781      0.692      0.867         42         141
  SZ      0.800      0.786      0.814          2           5

✔️ Zone co